#**Section 1: Libraries Setup**

In [ ]:
!pip install -q scikit-learn
!pip install -q transformers
!pip install -q tensorflow

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import confusion_matrix, classification_report
import itertools
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV3Large
from transformers import TFViTModel, ViTFeatureExtractor

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#**Section 2: Dataset Configurtion**

In [ ]:

import json
#token info
kaggle_token = {
    "username": "sarahdhainy",
    "key": "KGAT_a4866fa3ee103a6fda995d6f09b77c77"
}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_token, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("kaggle.json created successfully!")


kaggle.json created successfully!


In [ ]:
!pip install -q kaggle


In [ ]:
!kaggle datasets list


ref                                                       title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
wardabilal/spotify-global-music-dataset-20092025          Spotify Global Music Dataset (2009–2025)               1289021  2025-11-11 09:43:05.933000          10585        241  1.0              
rohiteng/amazon-sales-dataset                             Amazon Sales Dataset                                   4037578  2025-11-23 14:29:37.973000           3243         45  1.0              
khushikyad001/ai-impact-on-jobs-2030                      AI Impact on Jobs 2030                                   87410  2025-11-09 17:58:05.410000           6156        139  1.0              
mayabennett03/nba-historical-p

In [ ]:

import zipfile
import shutil
DATASET_SLUG= "grassknoted/asl-alphabet"
OUTPUT_ZIP= "/content/asl-alphabet.zip"

if not os.path.exists(OUTPUT_ZIP):
    print("Downloading ASL Alphabet dataset from Kaggle...")
    !kaggle datasets download -d {DATASET_SLUG} -p /content
else:
    print("Dataset zip already downloaded.")
EXTRACT_DIR = "/content/ASL_Alphabet_Dataset"
if os.path.exists(EXTRACT_DIR):
    print("Old dataset folder found — removing and re‑extracting.")
    shutil.rmtree(EXTRACT_DIR)

with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted to:", EXTRACT_DIR)
def flatten_if_double(path):
    entries = os.listdir(path)
    if len(entries)==1:
        inner = os.path.join(path, entries[0])
        if os.path.isdir(inner):
            for f in os.listdir(inner):
                shutil.move(os.path.join(inner,f), path)
            shutil.rmtree(inner)

for sub in ["asl_alphabet_train","asl_alphabet_test"]:
    folder = os.path.join(EXTRACT_DIR, sub)
    if os.path.isdir(folder):
        flatten_if_double(folder)
TRAIN_DIR = os.path.join(EXTRACT_DIR, "asl_alphabet_train")
TEST_DIR  = os.path.join(EXTRACT_DIR, "asl_alphabet_test")

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
print("Classes found:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR:", TEST_DIR)

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
 90% 948M/1.03G [00:00<00:00, 1.04GB/s]
100% 1.03G/1.03G [00:00<00:00, 1.19GB/s]
Dataset extracted to: /content/ASL_Alphabet_Dataset
Classes found: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Number of classes: 29
TRAIN_DIR: /content/ASL_Alphabet_Dataset/asl_alphabet_train
TEST_DIR: /content/ASL_Alphabet_Dataset/asl_alphabet_test


# **Section 3: Dataset Splitting + Preprocessing + Augmentation**

In [ ]:

import os
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_paths = []
labels = []

for class_name in CLASS_NAMES:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.endswith(('.png', '.jpg'))]
    image_paths.extend(files)
    labels.extend([class_name] * len(files))

image_paths = np.array(image_paths)
labels = np.array(labels)

total_images = len(image_paths)
print(f"Total images in dataset: {total_images}")

train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

print("\nDataset Split:")
print(f"Train+Validation: {len(train_val_paths)} images ({len(train_val_paths)/total_images*100:.2f}%)")
print(f"Test: {len(test_paths)} images ({len(test_paths)/total_images*100:.2f}%)")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_num = 1
for train_index, val_index in kf.split(train_val_paths):
    train_fold_paths = train_val_paths[train_index]
    val_fold_paths = train_val_paths[val_index]

    train_fold_labels = train_val_labels[train_index]
    val_fold_labels = train_val_labels[val_index]

    print(f"\nFold {fold_num}:")
    print(f"  Training: {len(train_fold_paths)} images ({len(train_fold_paths)/total_images*100:.2f}%)")
    print(f"  Validation: {len(val_fold_paths)} images ({len(val_fold_paths)/total_images*100:.2f}%)")

    fold_num += 1


IMG_SIZE = (200, 200)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_val_datagen = ImageDataGenerator(
    rescale=1./255
)


def create_generator(image_paths, labels, datagen):
    import tensorflow as tf
    import numpy as np
    from tensorflow.keras.utils import to_categorical

    class_indices = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}
    y = np.array([class_indices[label] for label in labels])

    def gen():
        for path, label in zip(image_paths, y):
            img = tf.keras.preprocessing.image.load_img(path, target_size=IMG_SIZE)
            img = tf.keras.preprocessing.image.img_to_array(img)
            img = datagen.random_transform(img)
            img = datagen.standardize(img)
            yield img, tf.keras.utils.to_categorical(label, NUM_CLASSES)

    dataset = tf.data.Dataset.from_generator(
        gen,
        output_types=(tf.float32, tf.float32),
        output_shapes=(IMG_SIZE + (3,), (NUM_CLASSES,))
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return dataset
train_dataset = create_generator(train_fold_paths, train_fold_labels, train_datagen)
val_dataset   = create_generator(val_fold_paths, val_fold_labels, test_val_datagen)
test_dataset  = create_generator(test_paths, test_labels, test_val_datagen)

print("\nData generators created successfully!")


Total images in dataset: 87000

Dataset Split:
Train+Validation: 69600 images (80.00%)
Test: 17400 images (20.00%)

Fold 1:
  Training: 55680 images (64.00%)
  Validation: 13920 images (16.00%)

Fold 2:
  Training: 55680 images (64.00%)
  Validation: 13920 images (16.00%)

Fold 3:
  Training: 55680 images (64.00%)
  Validation: 13920 images (16.00%)

Fold 4:
  Training: 55680 images (64.00%)
  Validation: 13920 images (16.00%)

Fold 5:
  Training: 55680 images (64.00%)
  Validation: 13920 images (16.00%)

Data generators created successfully!


#**Section 4: Building Models (CNN/ ResNet/ EfficientNet/ MobileNet)**

In [ ]:
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, GlobalAveragePooling2D
)

from tensorflow.keras.applications import (
    ResNet50, EfficientNetB0, MobileNetV3Large
)
INPUT_SIZE = (200, 200, 3)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
EPOCHS = 25
LOSS_FN = "categorical_crossentropy"
REGULARIZATION = "BatchNormalization + Dropout"
ACTIVATION = "ReLU"
FINAL_ACTIVATION = "Softmax"

print(f"""
MODEL HYPERPARAMETERS:
- Input size: {INPUT_SIZE}
- Batch size: {BATCH_SIZE}
- Epochs: {EPOCHS}
- Loss function: {LOSS_FN}
- Optimizer: Adam (lr={LEARNING_RATE})
- Regularization: {REGULARIZATION}
- Activation functions: {ACTIVATION}, Final = {FINAL_ACTIVATION}
""")

def build_custom_cnn(num_classes):
    model = models.Sequential([
        Input(shape=INPUT_SIZE),

        # Block 1
        Conv2D(32, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        # Block 2
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        # Block 3
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss=LOSS_FN,
        metrics=['accuracy']
    )
    return model
def build_tl_model(BaseModelClass, num_classes, freeze=True):
    base_model = BaseModelClass(
        include_top=False,
        weights='imagenet',
        input_shape=INPUT_SIZE
    )

    if freeze:
        for layer in base_model.layers:
            layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)

    optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        optimizer=optimizer,
        loss=LOSS_FN,
        metrics=['accuracy']
    )
    return model

cnn_model = build_custom_cnn(NUM_CLASSES)
resnet_model = build_tl_model(ResNet50, NUM_CLASSES)
efficientnet_model = build_tl_model(EfficientNetB0, NUM_CLASSES)
mobilenet_model = build_tl_model(MobileNetV3Large, NUM_CLASSES)



MODEL HYPERPARAMETERS:
- Input size: (200, 200, 3)
- Batch size: 32
- Epochs: 25
- Loss function: categorical_crossentropy
- Optimizer: Adam (lr=0.0001)
- Regularization: BatchNormalization + Dropout
- Activation functions: ReLU, Final = Softmax

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/applications/mobilenet_v3.py:517: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


#**Section 5: Training the Models**

In [ ]:
import shutil
from google.colab import files

import os, re, glob, pickle
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import KFold
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def find_latest_model(fold_path, model_name):
    pattern = os.path.join(fold_path, f"{model_name}_epoch*.keras")
    files = glob.glob(pattern)
    if not files:
        return None, 0

    def get_epoch_num(filename):
        m = re.search(r"_epoch(\d+)\.keras$", filename)
        return int(m.group(1)) if m else -1

    files_with_epochs = [(f, get_epoch_num(f)) for f in files]
    files_with_epochs = [x for x in files_with_epochs if x[1] >= 0]

    if not files_with_epochs:
        return None, 0

    latest_file = max(files_with_epochs, key=lambda x: x[1])
    return latest_file[0], latest_file[1]

class HistorySaver(Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.history_all = []

        if os.path.exists(filepath):
            try:
                with open(filepath, 'rb') as f:
                    self.history_all = pickle.load(f)
                print(f"[HistorySaver] Loaded {len(self.history_all)} previous epochs.")
            except:
                print("[HistorySaver] Could not load old history file. Starting new.")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.history_all.append(logs)

        with open(self.filepath, 'wb') as f:
            pickle.dump(self.history_all, f)

class ZipSaverCallback(Callback):
    def __init__(self, fold_path, model_name):
        super().__init__()
        self.fold_path = fold_path
        self.model_name = model_name

    def on_epoch_end(self, epoch, logs=None):
        zip_name = f"{self.model_name}_{os.path.basename(self.fold_path)}.zip"
        zip_path = os.path.join("/content", zip_name)


        if os.path.exists(zip_path):
            os.remove(zip_path)

        shutil.make_archive(zip_path.replace(".zip", ""), 'zip', self.fold_path)

        print(f"[ZipSaver] Exported fold backup ZIP → {zip_path}")
        files.download(zip_path)

def get_callbacks_all_epochs(model_name, fold):
    fold_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_fold{fold}")
    os.makedirs(fold_path, exist_ok=True)

    model_path = os.path.join(fold_path, f"{model_name}_epoch{{epoch:02d}}.keras")
    history_path = os.path.join(fold_path, "history.pkl")

    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),

        ModelCheckpoint(
            filepath=model_path,
            save_weights_only=False,
            save_best_only=False,
            verbose=1
        ),

        HistorySaver(history_path),
        ZipSaverCallback(fold_path, model_name)
    ]

def create_generator(paths, labels, datagen):
    df = pd.DataFrame({"filename": paths, "class": labels})

    return datagen.flow_from_dataframe(
        dataframe=df,
        x_col="filename",
        y_col="class",
        target_size=INPUT_SIZE[:2],
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=True
    )

def train_model_on_folds_all_epochs(build_model_fn, image_paths, labels, model_name):
    histories = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(image_paths), start=1):
        print(f"\n==============================")
        print(f"== {model_name} — Fold {fold}/5 ==")
        print(f"==============================\n")

        fold_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_fold{fold}")
        os.makedirs(fold_path, exist_ok=True)

        history_file = os.path.join(fold_path, "history.pkl")
        train_paths = image_paths[train_idx]
        val_paths   = image_paths[val_idx]
        train_l     = labels[train_idx]
        val_l       = labels[val_idx]

        train_gen = create_generator(train_paths, train_l, train_datagen)
        val_gen   = create_generator(val_paths, val_l, test_val_datagen)

        latest_model_file, latest_epoch = find_latest_model(fold_path, model_name)

        if latest_model_file:
            print(f"[Resume] Found model: {latest_model_file}")
            model = tf.keras.models.load_model(latest_model_file)
            initial_epoch = latest_epoch
            print(f"[Resume] Continuing from epoch {initial_epoch}")
        else:
            print("[Resume] No checkpoint found — starting new fold.")
            model = build_model_fn()
            initial_epoch = 0
        if initial_epoch >= EPOCHS:
            print(f"[Skip] Fold already complete ({initial_epoch} epochs).")
            continue



        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            initial_epoch=initial_epoch,
            callbacks=get_callbacks_all_epochs(model_name, fold),
            verbose=1
        )

        histories.append(history)

    return histories



In [ ]:
cnn_histories = train_model_on_folds_all_epochs(
     lambda: build_custom_cnn(NUM_CLASSES),
     train_val_paths,
     train_val_labels,
    "CustomCNN"
)


In [ ]:
resnet_histories = train_model_on_folds_all_epochs(
    lambda: build_tl_model(ResNet50, NUM_CLASSES),
    train_val_paths,
    train_val_labels,
    "ResNet50"
)

In [ ]:
efficientnet_histories = train_model_on_folds_all_epochs(
    lambda: build_tl_model(EfficientNetB0, NUM_CLASSES),
    train_val_paths,
    train_val_labels,
    "EfficientNetB0"
)


In [ ]:
mobilenet_histories = train_model_on_folds_all_epochs(
    lambda: build_tl_model(MobileNetV3Large, NUM_CLASSES),
    train_val_paths,
    train_val_labels,
    "MobileNetV3Large"
)

#**Section 6: Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def evaluate_model(model, test_dataset, model_name="Model"):
    print(f"\nEvaluating {model_name}...")

    y_true = []
    y_pred = []

    for images, labels in test_dataset:
        preds = model.predict(images)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
    print(f"\nClassification Report for {model_name}:\n", report)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12,10))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    return report, cm

report_cnn, cm_cnn = evaluate_model(cnn_model, test_dataset, model_name="Custom CNN")
report_resnet, cm_resnet = evaluate_model(resnet_model, test_dataset, model_name="ResNet50")
report_efficient, cm_efficient = evaluate_model(efficientnet_model, test_dataset, model_name="EfficientNetB0")
report_mobilenet, cm_mobilenet = evaluate_model(mobilenet_model, test_dataset, model_name="MobileNetV3Large")


def compare_train_val(history, model_name="Model"):
    print(f"\nPerformance Summary for {model_name}:")
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]

    print(f"Final Train Accuracy: {final_train_acc:.4f}, Final Validation Accuracy: {final_val_acc:.4f}")
    print(f"Final Train Loss: {final_train_loss:.4f}, Final Validation Loss: {final_val_loss:.4f}")

compare_train_val(history_cnn, "Custom CNN")
compare_train_val(history_resnet, "ResNet50")
compare_train_val(history_efficient, "EfficientNetB0")
compare_train_val(history_mobilenet, "MobileNetV3Large")


#**Section 7: Testing**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image


def load_and_preprocess_image(img_path, img_size):
    img = image.load_img(img_path, target_size=(img_size, img_size))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array, img


def predict_single_image(model, img_path, model_name="Model", img_size=IMG_SIZE):
    print(f"\nRunning final test on {model_name}...")

    img_array, original_img = load_and_preprocess_image(img_path, img_size)
    preds= model.predict(img_array)
    pred_index = np.argmax(preds)
    pred_label = CLASS_NAMES[pred_index]


    plt.imshow(original_img)
    plt.title(f"Predicted: {pred_label}")
    plt.axis("off")
    plt.show()

    return pred_label


test_image_path="/content/ASL_Alphabet_Dataset/asl_alphabet_test/A_test.jpg"

predict_single_image(cnn_model,test_image_path, model_name="Custom CNN",img_size=IMG_SIZE)
predict_single_image(resnet_model,test_image_path, model_name="ResNet50",img_size=IMG_SIZE)
predict_single_image(efficientnet_model,test_image_path, model_name="EfficientNetB0",img_size=IMG_SIZE)
predict_single_image(mobilenet_model, test_image_path, model_name="MobileNetV3Large",img_size=IMG_SIZE)
